# Privacy Threat Model for ML Systems

ML models trained on sensitive data can leak information about their training set. This notebook maps the attack surface, establishes the threat model, and motivates the privacy-preserving methods covered in subsequent notebooks.

## Privacy Attacks on ML Models

**Membership inference attack** (Shokri et al. 2017): given a model and a data point, determine whether that point was in the training set. Models overfit to training data, producing higher confidence on training examples — this signal enables inference.

**Attribute inference**: given public attributes and a model, infer sensitive attributes of training records. A model trained to predict income may reveal demographic attributes of individuals in its training data.

**Model inversion** (Fredrikson et al. 2015): given model access, reconstruct features of training data. For face recognition models, this can reconstruct recognizable faces.

**Data extraction** (Carlini et al. 2020): LLMs memorize and can reproduce verbatim training data including PII, secrets, and copyrighted content. Provably shown by eliciting memorized sequences via specific prompts.

The severity of each attack depends on model architecture, training data sensitivity, and the model's degree of memorization.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random, math

@dataclass
class PrivacyRisk:
    attack_type: str
    likelihood: str  # 'low', 'medium', 'high'
    impact: str
    mitigations: list

def privacy_audit(model_description: dict) -> list:
    risks = []
    # Membership inference risk
    mia_likelihood = 'high' if model_description.get('overfit', False) else 'medium'
    risks.append(PrivacyRisk(
        attack_type='membership_inference',
        likelihood=mia_likelihood,
        impact='Determines if individual was in training set',
        mitigations=['differential_privacy', 'regularization', 'early_stopping']
    ))
    if model_description.get('sensitive_attributes', []):
        risks.append(PrivacyRisk(
            attack_type='attribute_inference',
            likelihood='high',
            impact=f'Infers {model_description["sensitive_attributes"]}',
            mitigations=['fairness_constraints', 'differential_privacy', 'data_minimization']
        ))
    if model_description.get('is_generative', False):
        risks.append(PrivacyRisk(
            attack_type='data_extraction',
            likelihood='high',
            impact='Verbatim reproduction of training data including PII',
            mitigations=['deduplication', 'differential_privacy', 'output_filtering']
        ))
    if model_description.get('trained_on_images', False):
        risks.append(PrivacyRisk(
            attack_type='model_inversion',
            likelihood='medium',
            impact='Reconstruction of training images',
            mitigations=['differential_privacy', 'output_perturbation']
        ))
    return risks

model_desc = {
    'name': 'Medical Diagnosis Classifier',
    'overfit': True,
    'sensitive_attributes': ['diagnosis', 'age', 'genetic_markers'],
    'is_generative': False,
    'trained_on_images': True,
}
risks = privacy_audit(model_desc)
print(f'Privacy audit for: {model_desc["name"]}')
print(f'Total risks identified: {len(risks)}')
for r in risks:
    print(f'\n[{r.likelihood.upper()} RISK] {r.attack_type}')
    print(f'  Impact: {r.impact}')
    print(f'  Mitigations: {r.mitigations}')

## Membership Inference Attack

A simple membership inference attack exploits the confidence gap between training and test examples. Models output higher confidence on training examples because they have memorized them.

In [ ]:
def simulate_membership_inference(model_fn: Callable, train_data: list, test_data: list,
                                    threshold: float = 0.75) -> dict:
    train_preds = [model_fn(x, is_member=True) for x in train_data]
    test_preds = [model_fn(x, is_member=False) for x in test_data]
    tp = sum(1 for p in train_preds if p['confidence'] > threshold)
    fp = sum(1 for p in test_preds if p['confidence'] > threshold)
    tn = len(test_preds) - fp
    fn = len(train_preds) - tp
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall = tp / (tp + fn) if (tp+fn) > 0 else 0
    advantage = tp/len(train_preds) - fp/len(test_preds)
    return {
        'threshold': threshold, 'precision': precision, 'recall': recall,
        'advantage': advantage, 'tp': tp, 'fp': fp,
        'interpretation': 'significant leakage' if advantage > 0.2 else 'moderate leakage' if advantage > 0.05 else 'low leakage'
    }

rng = random.Random(42)
def overfit_model(x, is_member=False):
    base = 0.9 if is_member else 0.6
    return {'confidence': min(1.0, max(0, base + rng.gauss(0, 0.1)))}

train_data = list(range(100))
test_data = list(range(100, 200))
result = simulate_membership_inference(overfit_model, train_data, test_data)
print('Membership Inference Attack Results:')
for k, v in result.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

## Privacy Budget and the Defense Landscape

Privacy-preserving ML techniques sit on a spectrum:

**Differential Privacy (DP)**: provides formal, quantifiable privacy guarantees. Every trained model has a privacy budget ε — lower ε means stronger privacy. Training with DP-SGD adds calibrated noise during training.

**Federated Learning (FL)**: train on distributed data without centralizing it. Data stays on device; only model updates are shared. Reduces exposure but does not provide DP guarantees unless combined with DP-SGD.

**Secure Aggregation**: uses cryptographic protocols so the server learns only the aggregate of client updates, not individual updates. Complements FL.

**Data minimization and anonymization**: pre-training data hygiene — reduce PII, deduplicate, apply k-anonymity. Necessary but not sufficient.

These techniques are complementary: a production privacy-preserving ML system typically uses FL + DP-SGD + secure aggregation together.